# HDAT DS 독립 모의실습 — Process 8단계 (Solution)

> 현대자동차그룹 또는 현대엔지비의 공식·복원 문제가 아닌 **독립 창작 연습문제**입니다. PyTorch만 사용합니다.

Starter의 8개 계약을 한 가지 정석 풀이로 구현합니다. 정답을 외우기보다 각 `assert`가 어떤 실수를 막는지 설명할 수 있는지 확인하세요.

## 실행 환경과 시험 습관

- 권장: Python 3.10–3.12, PyTorch 2.2 이상, JupyterLab/Notebook 7 이상
- CPU에서 수 초 안에 실행됩니다.
- 시작 전 **Kernel → Restart Kernel and Clear Outputs**, 큰 단계 후 **Ctrl/Cmd+S**를 누르세요.
- 해설을 본 뒤에는 출력물을 지우고 위에서 아래로 다시 실행해 숨은 변수 의존성을 확인하세요.

In [ ]:
import platform
import random
import tempfile
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

print('Python :', platform.python_version(), '(권장 3.10–3.12)')
print('PyTorch:', torch.__version__, '(권장 2.2+)')
SEED = 2026
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cpu')
print('device :', device)

## 문제 데이터

센서 특징 4개로 정상(0)/이상(1)을 예측하는 작은 합성 데이터입니다. 원본의 dtype과 레이블 차원을 일부러 모델 계약과 다르게 만들었습니다.

In [ ]:
g = torch.Generator().manual_seed(SEED)
raw_X = torch.randn(40, 4, generator=g, dtype=torch.float64)
signal = 1.4 * raw_X[:, 0] - 0.9 * raw_X[:, 1] + 0.5 * raw_X[:, 2]
raw_y = (signal + 0.25 * torch.randn(40, generator=g) > 0).to(torch.int64)
print('raw_X:', raw_X.shape, raw_X.dtype)
print('raw_y:', raw_y.shape, raw_y.dtype, 'positive rate=', raw_y.float().mean().item())

## 1. 텐서 계약 맞추기

손실함수와 모델이 기대하는 계약을 먼저 고정하면 뒤에서 발생하는 오류가 크게 줄어듭니다. `unsqueeze(1)`은 `[N]`을 `[N, 1]`로 바꿉니다.

In [ ]:
X = raw_X.to(torch.float32)
y = raw_y.to(torch.float32).unsqueeze(1)

assert isinstance(X, torch.Tensor) and isinstance(y, torch.Tensor)
assert X.shape == (40, 4) and y.shape == (40, 1)
assert X.dtype == torch.float32 and y.dtype == torch.float32
print('X/y contract:', X.shape, X.dtype, y.shape, y.dtype)

## 2. 순서를 보존한 train/validation 분할

문제 지시가 순서 보존이라면 임의 분할 함수를 쓰지 말고 경계를 명시합니다. 실제 시계열에서는 특히 미래가 train에 섞이지 않아야 합니다.

In [ ]:
split_at = 32
X_train, y_train = X[:split_at], y[:split_at]
X_val, y_val = X[split_at:], y[split_at:]

assert X_train.shape == (32, 4) and y_train.shape == (32, 1)
assert X_val.shape == (8, 4) and y_val.shape == (8, 1)
assert torch.equal(X_train, X[:32]) and torch.equal(X_val, X[32:])

## 3. 데이터 누수 없는 표준화

validation 통계까지 사용하면 실제 배포에서 알 수 없는 미래 정보를 본 셈입니다. train에서 구한 `mu`, `sigma`를 그대로 재사용합니다.

In [ ]:
mu = X_train.mean(dim=0, keepdim=True)
sigma = X_train.std(dim=0, keepdim=True, unbiased=False).clamp_min(1e-6)
X_train_z = (X_train - mu) / sigma
X_val_z = (X_val - mu) / sigma

assert mu.shape == (1, 4) and sigma.shape == (1, 4)
assert X_train_z.shape == X_train.shape and X_val_z.shape == X_val.shape
assert torch.all(sigma > 0)
assert torch.allclose(X_train_z.mean(0), torch.zeros(4), atol=1e-5)

## 4. Dataset과 DataLoader

학습 데이터만 섞습니다. 별도 `Generator`에 seed를 주면 shuffle 순서를 다시 만들 수 있습니다.

In [ ]:
train_ds = TensorDataset(X_train_z, y_train)
val_ds = TensorDataset(X_val_z, y_val)
loader_g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, generator=loader_g)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

xb, yb = next(iter(train_loader))
assert xb.shape == (8, 4) and yb.shape == (8, 1)
assert xb.dtype == torch.float32 and yb.dtype == torch.float32
assert len(train_ds) == 32 and len(val_ds) == 8

## 5. 작은 `nn.Module`

`BCEWithLogitsLoss`가 내부에서 안정적인 sigmoid 계산을 하므로 모델 끝에는 sigmoid를 넣지 않습니다. 출력은 `[B, 1]` logit입니다.

In [ ]:
class TinyBinaryClassifier(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.net(x)

model = TinyBinaryClassifier(n_features=4).to(device)
test_logits = model(torch.zeros(3, 4))
assert test_logits.shape == (3, 1) and test_logits.dtype == torch.float32

## 6. 한 번의 학습 step

gradient 누적을 피하려면 매 step 시작에 `zero_grad`가 필요합니다. `set_to_none=True`는 효율적이며 누락된 gradient도 찾기 쉽습니다.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
xb, yb = next(iter(train_loader))
optimizer.zero_grad(set_to_none=True)
logits = model(xb)
loss = criterion(logits, yb)
loss.backward()
optimizer.step()

assert isinstance(loss, torch.Tensor) and loss.ndim == 0
assert torch.isfinite(loss).item()
assert all(p.grad is not None for p in model.parameters() if p.requires_grad)
print('one-step loss:', round(loss.item(), 4))

## 7. 학습 loop와 validation

학습 때는 `train`, 평가 때는 `eval`을 호출합니다. 평가에서는 gradient graph가 필요 없으므로 `torch.inference_mode()`로 감쌉니다.

In [ ]:
for epoch in range(30):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

model.eval()
with torch.inference_mode():
    val_logits = model(X_val_z)
    val_pred = (val_logits >= 0).to(y_val.dtype)
    val_accuracy = (val_pred == y_val).float().mean().item()

assert isinstance(val_accuracy, float)
assert 0.0 <= val_accuracy <= 1.0
print(f'validation accuracy: {val_accuracy:.3f}')

## 8. `state_dict` 저장·재로딩·예측 동일성

모델 구조는 코드로 다시 만들고 학습된 파라미터만 저장합니다. 재로딩 뒤 같은 입력에 같은 확률이 나오면 최소한의 제출 전 검증을 통과한 것입니다.

In [ ]:
tmp_dir = tempfile.TemporaryDirectory()
checkpoint_path = Path(tmp_dir.name) / 'tiny_model.pt'
torch.save({'model_state': model.state_dict(), 'n_features': 4}, checkpoint_path)

try:
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
except TypeError:  # PyTorch 구버전 호환
    checkpoint = torch.load(checkpoint_path, map_location='cpu')

reloaded = TinyBinaryClassifier(checkpoint['n_features'])
reloaded.load_state_dict(checkpoint['model_state'])
reloaded.eval()
with torch.inference_mode():
    original_prob = torch.sigmoid(model(X_val_z))
    reloaded_prob = torch.sigmoid(reloaded(X_val_z))

assert checkpoint_path.exists()
assert isinstance(reloaded, TinyBinaryClassifier)
assert original_prob.shape == (8, 1)
assert torch.allclose(original_prob, reloaded_prob, atol=1e-7)
print('8개 계약 완료 — 저장 후 Kernel Restart + Run All로 다시 확인하세요.')

## 최종 점검표

- [x] 입력과 레이블의 shape/dtype을 초기에 고정했다.
- [x] train 통계만으로 validation을 변환했다.
- [x] 모델은 logit을 반환하고 손실함수가 sigmoid를 담당한다.
- [x] validation에서 `eval`과 inference 문맥을 사용했다.
- [x] `state_dict`를 새 모델에 재로딩하고 예측 동일성을 확인했다.